# Chapter 25 — Credit Derivatives and the CDS Market

Every number in the chapter, from the ground-up valuation to the convertible on a tree with default and the hybrid note.

| Section | What it reproduces | Example |
|---|---|---|
| 1 | A CDS valued from its quote, and the par spread recovered | 25.1 |
| 2 | Spread to upfront, and why the annuity shortens | 25.2 |
| 3 | Marking a position: the annuity moves too | 25.3 |
| 4 | An index trades at the annuity-weighted average | 25.4 |
| 5 | One portfolio, five tranches, one correlation | 25.5 |
| 6 | The same five issuers as five basket instruments | 25.6 |
| 7 | What the restructuring clause is worth | 25.7 |
| 8 | Decomposing a convertible | 25.8 |
| 9 | The convertible on a tree with default: conversion, the issuer's call, and Merton's jump-to-ruin | 25.9 |
| 10 | The correlation a hybrid needs and no market quotes | 25.10 |

All of it is deterministic — there is no simulation and no seed anywhere. Sections 5
and 6 integrate a one-factor Gaussian copula over the common factor by numerical
integration on a fine grid rather than by sampling, so they return the same digits every run.

Column headers and the labels passed to `show()` stay in English: CJK characters are
double-width in a monospace terminal while Python's `%-50s` counts characters, so
translating one would misalign every row beneath it.


In [1]:
import numpy as np
from math import exp, log, sqrt, tanh
from scipy.stats import norm, binom

N = norm.cdf
R, RATE, FREQ, TENOR = 0.40, 0.03, 4, 5.0   # recovery, discount rate, premiums a year, tenor
NOTIONAL = 10_000_000.0


def show(label, got, printed, tol=6e-4):
    """Print a computed value beside the one the chapter prints."""
    flag = "ok" if abs(got - printed) <= tol * max(1.0, abs(printed)) else "DIFFERS"
    print("%-46s %13.4f   book %11.4f   %s" % (label, got, printed, flag))


def cds_legs(lam, T=TENOR, R=R, r=RATE, freq=FREQ, accrual=True, pay_at_default=False):
    """Risky annuity and protection leg under a flat hazard rate.

    Returns (annuity, protection, scheduled, accrual). The chapter's convention
    discounts the default payment to the end of the quarter in which it arrives;
    pay_at_default moves it to the middle of the quarter, which is what the
    contract actually says and is worth a few hundredths of a percent.
    """
    n, dt = int(round(T * freq)), 1.0 / freq
    sched = accr = prot = 0.0
    for i in range(1, n + 1):
        t, t_prev = i * dt, (i - 1) * dt
        Q, Q_prev = exp(-lam * t), exp(-lam * t_prev)
        dQ = Q_prev - Q
        df = exp(-r * t)
        sched += dt * df * Q
        if accrual:
            accr += 0.5 * dt * df * dQ
        prot += (exp(-r * (t - dt / 2)) if pay_at_default else df) * dQ
    return sched + accr, (1 - R) * prot, sched, accr


## 1. A CDS from the ground up — Example 25.1

\$10 million of five-year protection on an issuer quoted at 250 bp, traded against the
standard 100 bp coupon, recovery 40%, rates 3%.

The credit triangle turns the quote into a hazard rate; the two legs follow; and the
par spread comes back out. **That it comes back almost exactly is a property of the
convention, not a theorem** — the accrual term is what makes it work, and it is worth
only 0.0218 of the 4.1865.


In [2]:
s_par, coupon = 0.0250, 0.0100
lam = s_par / (1 - R)
print("hazard rate  lambda = s/(1-R) = %.3f%%" % (100 * lam))
print()
print("%6s %10s %22s" % ("year", "Q(t)", "cumulative default"))
for t in (1, 2, 3, 5):
    Q = exp(-lam * t)
    print("%6d %10.4f %21.2f%%" % (t, Q, 100 * (1 - Q)))
print()

A, prot, sched, accr = cds_legs(lam)
show("scheduled premium leg", sched, 4.1647)
show("accrual on default", accr, 0.0218)
show("risky annuity A", A, 4.1865)
show("protection leg, % of notional", 100 * prot, 10.4662)
show("par spread recovered, bp", 1e4 * prot / A, 250.00)
print()

dt = 1.0 / FREQ
show("closed form 2(1-R)/dt tanh(lam dt/2), bp",
     1e4 * 2 * (1 - R) / dt * tanh(lam * dt / 2), 249.998)
_, prot_d, _, _ = cds_legs(lam, pay_at_default=True)
show("protection paid at default, % of notional", 100 * prot_d, 10.5056)
print()

upfront = 100 * prot - 100 * coupon * A
show("upfront, legs differenced, %", upfront, 6.2797)
show("upfront, short form (s-c)A, %", 100 * (s_par - coupon) * A, 6.2798)
print("on $10 million that is $%s" % format(NOTIONAL * upfront / 100, ",.0f"))


hazard rate  lambda = s/(1-R) = 4.167%

  year       Q(t)     cumulative default
     1     0.9592                  4.08%
     2     0.9200                  8.00%
     3     0.8825                 11.75%
     5     0.8119                 18.81%

scheduled premium leg                                 4.1647   book      4.1647   ok
accrual on default                                    0.0218   book      0.0218   ok
risky annuity A                                       4.1865   book      4.1865   ok
protection leg, % of notional                        10.4662   book     10.4662   ok
par spread recovered, bp                            249.9977   book    250.0000   ok

closed form 2(1-R)/dt tanh(lam dt/2), bp            249.9977   book    249.9980   ok
protection paid at default, % of notional            10.5055   book     10.5056   ok

upfront, legs differenced, %                          6.2797   book      6.2797   ok
upfront, short form (s-c)A, %                         6.2798   book     

## 2. From a spread to an upfront — Example 25.2

**Watch the annuity, not the upfront.** It falls from 4.55 to 3.40 as the spread widens,
because a contract on a deteriorating credit is expected to terminate early — the
premium stream is not expected to run its full five years. That shortening is why the
upfront is not simply linear in the spread, and it is the same effect that Section 3
catches a desk getting wrong.


In [3]:
print("%12s %14s %16s %12s" % ("par spread", "hazard rate", "risky annuity", "upfront"))
for s_bp in (40, 100, 150, 250, 500, 800):
    s = s_bp / 1e4
    lam_s = s / (1 - R)
    A_s, _, _, _ = cds_legs(lam_s)
    print("%9d bp %13.3f%% %16.4f %11.2f%%"
          % (s_bp, 100 * lam_s, A_s, 100 * (s - coupon) * A_s))
print()
a_lo, _, _, _ = cds_legs(0.0040 / (1 - R))
a_hi, _, _, _ = cds_legs(0.0800 / (1 - R))
print("the annuity shortens from %.2f to %.2f as the spread goes 40 bp -> 800 bp"
      % (a_lo, a_hi))


  par spread    hazard rate    risky annuity      upfront
       40 bp         0.667%           4.5513       -2.73%
      100 bp         1.667%           4.4428        0.00%
      150 bp         2.500%           4.3550        2.18%
      250 bp         4.167%           4.1865        6.28%
      500 bp         8.333%           3.8027       15.21%
      800 bp        13.333%           3.4044       23.83%

the annuity shortens from 4.55 to 3.40 as the spread goes 40 bp -> 800 bp


## 3. The annuity moves too — Example 25.3

A desk bought five-year protection at 120 bp on the standard 100 bp coupon; a year later the four-year spread is 300 bp.
The mark is $(s_t - c)A_t$ with $c$ the 100 bp coupon, not the 120 bp it traded at — **both objects carry the subscript $t$.** Using the annuity at
the old spread overstates the gain, and the error is systematic in one direction: as a
credit deteriorates the annuity shortens, so protection gains *less* than a naive
duration calculation suggests.


In [4]:
c = 0.0100                                   # the standard coupon the contract pays
A_new, _, _, _ = cds_legs(0.0300 / (1 - R), T=4.0)
A_old, _, _, _ = cds_legs(0.0120 / (1 - R), T=4.0)
mtm_new = (0.0300 - c) * A_new
mtm_old = (0.0300 - c) * A_old
mtm_traded = (0.0300 - 0.0120) * A_new

show("four-year annuity at the new spread", A_new, 3.4103)
show("MTM on the current curve, % of notional", 100 * mtm_new, 6.82)
show("MTM using the old-spread annuity, %", 100 * mtm_old, 7.22)
show("overstatement, %", 100 * (mtm_old / mtm_new - 1), 5.9, tol=3e-3)
show("marking against the traded 120 bp, %", 100 * mtm_traded, 6.14)
print()
print("on $10 million the correct mark is $%s" % format(NOTIONAL * mtm_new, ",.0f"))


four-year annuity at the new spread                   3.4103   book      3.4103   ok
MTM on the current curve, % of notional               6.8207   book      6.8200   ok
MTM using the old-spread annuity, %                   7.2236   book      7.2200   ok
overstatement, %                                      5.9076   book      5.9000   ok
marking against the traded 120 bp, %                  6.1386   book      6.1400   ok

on $10 million the correct mark is $682,067


## 4. Why the index is not the average — Example 25.4

125 issuers averaging 80 bp: 115 at 55 bp and 10 distressed at 367.5 bp. The index does
not trade at 80. **A portfolio instrument prices at the annuity-weighted average of its
constituents**, and a wide-spread issuer has a shorter annuity, so it carries less weight than
its spread suggests. The residual against the traded level is the index skew; the 2.7 bp
here is arithmetic, not a mispricing.


In [5]:
BUCKETS = [(115, 0.0055), (10, 0.03675)]

weighted = total_annuity = 0.0
for count, s in BUCKETS:
    A_j, _, _, _ = cds_legs(s / (1 - R))
    weighted += count * A_j * s
    total_annuity += count * A_j

arithmetic = sum(count * s for count, s in BUCKETS) / sum(c for c, _ in BUCKETS)
show("arithmetic average, bp", 1e4 * arithmetic, 80.0)
show("annuity-weighted index spread, bp", 1e4 * weighted / total_annuity, 77.3)
print()
for count, s in BUCKETS:
    A_j, _, _, _ = cds_legs(s / (1 - R))
    print("%4d issuers at %6.1f bp carry a risky annuity of %.4f"
          % (count, 1e4 * s, A_j))


arithmetic average, bp                               80.0000   book     80.0000   ok
annuity-weighted index spread, bp                    77.3104   book     77.3000   ok

 115 issuers at   55.0 bp carry a risky annuity of 4.5239
  10 issuers at  367.5 bp carry a risky annuity of 3.9997


## 5. One portfolio, five tranches — Example 25.5

125 issuers, each with a 7% five-year default probability and 40% recovery. **The
portfolio's expected loss is 4.200% at every correlation** — correlation redistributes
loss between the layers and creates none.

Read down the columns and the two ends move in opposite directions. The senior tranche
is where the danger lives: valued on $\rho = 0.10$ and actually facing $\rho = 0.50$, its
expected loss is **43 times larger**, and nothing about the portfolio has changed.


In [6]:
NAMES, PD5 = 125, 0.07
LOSS_PER_DEFAULT = (1 - R) / NAMES
TRANCHES = [(0.00, 0.03), (0.03, 0.07), (0.07, 0.10), (0.10, 0.15), (0.15, 0.30)]

# A fine grid over the common factor, weighted by the standard normal density.
# Gauss-Hermite is the textbook choice and is wrong here: its weights overflow
# past a couple of hundred nodes, and below that the senior tranche -- a tail
# of a tail -- has not converged. This grid is converged by a thousand points.
_x = np.linspace(-8.0, 8.0, 4001)
_w = norm.pdf(_x)
_w = _w / _w.sum()
_k = np.arange(NAMES + 1)
_losses = _k * LOSS_PER_DEFAULT


def default_pmf(rho, p=PD5):
    """Unconditional distribution of the number of defaults, one-factor Gaussian."""
    if rho <= 0.0:
        return binom.pmf(_k, NAMES, p)
    c = norm.ppf(p)
    p_m = N((c - sqrt(rho) * _x) / sqrt(1 - rho))
    return (binom.pmf(_k[:, None], NAMES, p_m[None, :]) * _w[None, :]).sum(axis=1)


def tranche_el(pmf, a, b):
    """Expected loss as a fraction of the tranche's own notional."""
    return (pmf * np.clip(_losses - a, 0.0, b - a)).sum() / (b - a)


print("%6s %10s %10s %10s %10s %10s %12s"
      % ("rho", "0-3%", "3-7%", "7-10%", "10-15%", "15-30%", "portfolio"))
for rho in (0.00, 0.10, 0.20, 0.30, 0.50, 0.80):
    pmf = default_pmf(rho)
    row = [100 * tranche_el(pmf, a, b) for a, b in TRANCHES]
    portfolio = 100 * (pmf * _losses).sum()
    print("%6.2f %9.2f%% %9.2f%% %9.2f%% %9.2f%% %9.2f%% %11.3f%%"
          % (rho, row[0], row[1], row[2], row[3], row[4], portfolio))
print()

print("%6s %24s %20s" % ("rho", "15-30% expected loss", "P(any loss at all)"))
for rho in (0.10, 0.30, 0.50, 0.80):
    pmf = default_pmf(rho)
    el = 100 * tranche_el(pmf, 0.15, 0.30)
    p_any = 100 * pmf[_losses > 0.15].sum()
    print("%6.2f %23.3f%% %19.2f%%" % (rho, el, p_any))
print()
el_10 = tranche_el(default_pmf(0.10), 0.15, 0.30)
el_50 = tranche_el(default_pmf(0.50), 0.15, 0.30)
print("valued at rho = 0.10 and facing rho = 0.50, the senior tranche's expected")
print("loss is %.0f times larger, and nothing about the portfolio has changed"
      % (el_50 / el_10))


   rho       0-3%       3-7%      7-10%     10-15%     15-30%    portfolio
  0.00     95.76%     32.73%      0.59%      0.00%      0.00%       4.200%
  0.10     81.81%     33.49%      9.44%      2.16%      0.10%       4.200%
  0.20     71.40%     31.51%     13.50%      5.48%      0.78%       4.200%
  0.30     62.59%     29.30%     15.21%      7.87%      1.88%       4.200%
  0.50     47.36%     24.75%     15.81%     10.40%      4.27%       4.200%
  0.80     26.59%     17.11%     13.35%     10.75%      6.96%       4.200%

   rho     15-30% expected loss   P(any loss at all)


  0.10                   0.098%                0.61%
  0.30                   1.877%                4.97%
  0.50                   4.272%                7.93%
  0.80                   6.962%                9.44%



valued at rho = 0.10 and facing rho = 0.50, the senior tranche's expected
loss is 43 times larger, and nothing about the portfolio has changed


## 6. Five issuers, five instruments — Example 25.6

Five issuers each at 150 bp. **At zero correlation the first-to-default costs the sum of
its parts** — five issuers at 150 give 750 — because with independent issuers the first default
arrives at the sum of their hazard rates, so the basket is a CDS on one issuer five times as risky. As correlation rises the seesaw of Section 5 reappears in a
different instrument: the first-to-default falls and the fifth-to-default rises.

And the five add up. Since the number of defaults is the sum over $k$ of the indicator
that at least $k$ have occurred, the five protection legs must sum to five single-name
legs at every correlation — the last row is that check.


In [7]:
BASKET, S_NAME = 5, 0.0150
LAM_NAME = S_NAME / (1 - R)
_j = np.arange(BASKET + 1)


def at_least_k_by(t, rho, k):
    """P(at least k of the five have defaulted by t), one-factor Gaussian copula."""
    F = 1.0 - exp(-LAM_NAME * t)
    if rho <= 0.0:
        return binom.sf(k - 1, BASKET, F)
    c = norm.ppf(F)
    p_m = N((c - sqrt(rho) * _x) / sqrt(1 - rho))
    return float((_w * binom.sf(k - 1, BASKET, p_m)).sum())


def basket_spread(rho, k, T=TENOR, r=RATE, freq=FREQ):
    """Par spread of the k-th-to-default basket, same conventions as a single-name CDS."""
    n, dt = int(round(T * freq)), 1.0 / freq
    ann = prot = 0.0
    for i in range(1, n + 1):
        t, t_prev = i * dt, (i - 1) * dt
        G, G_prev = at_least_k_by(t, rho, k), at_least_k_by(t_prev, rho, k)
        df = exp(-r * t)
        dG = G - G_prev
        ann += dt * df * (1.0 - G) + 0.5 * dt * df * dG
        prot += df * dG
    return (1 - R) * prot / ann, (1 - R) * prot


print("%6s %8s %8s %8s %8s %8s" % ("rho", "1st", "2nd", "3rd", "4th", "5th"))
for rho in (0.00, 0.15, 0.30, 0.60, 0.90):
    spreads = [basket_spread(rho, k)[0] for k in range(1, BASKET + 1)]
    print("%6.2f %8.0f %8.0f %8.0f %8.0f %8.0f"
          % (rho, *[1e4 * s for s in spreads]))
print()

_, single_prot, _, _ = cds_legs(LAM_NAME)
print("%6s %22s %22s" % ("rho", "sum of basket legs", "5 single-name legs"))
for rho in (0.00, 0.30, 0.90):
    total = sum(basket_spread(rho, k)[1] for k in range(1, BASKET + 1))
    print("%6.2f %21.6f%% %21.6f%%" % (rho, 100 * total, 100 * BASKET * single_prot))


   rho      1st      2nd      3rd      4th      5th
  0.00      750      133       16        1        0
  0.15      669      160       34        5        0
  0.30      591      179       56       15        2


  0.60      439      196       99       47       17
  0.90      269      185      138      102       67

   rho     sum of basket legs     5 single-name legs
  0.00             32.662700%             32.662700%
  0.30             32.662700%             32.662700%


  0.90             32.662700%             32.662700%


## 7. What the restructuring clause is worth — Example 25.7

A 3% annual intensity of bankruptcy or failure to pay, recovering 40%; including restructuring adds
15% to that intensity, but restructuring events recover 70%.

**The point is not the number, which depends entirely on the assumed intensities.** It is
that the number is small and stable right up until it is neither: when a restructuring is
actually on the table, the two contracts stop being near-substitutes.


In [8]:
LAM_HARD, R_HARD = 0.03, 0.40
LAM_RESTR, R_RESTR = 0.03 * 0.15, 0.70

s_no = LAM_HARD * (1 - R_HARD)
s_with = s_no + LAM_RESTR * (1 - R_RESTR)

show("no-restructuring contract, bp", 1e4 * s_no, 180.0)
show("with restructuring, bp", 1e4 * s_with, 193.5)
show("value of the clause, bp", 1e4 * (s_with - s_no), 13.5)
show("as a share of the spread, %", 100 * (s_with - s_no) / s_with, 7.0, tol=5e-3)


no-restructuring contract, bp                       180.0000   book    180.0000   ok
with restructuring, bp                              193.5000   book    193.5000   ok
value of the clause, bp                              13.5000   book     13.5000   ok
as a share of the spread, %                           6.9767   book      7.0000   ok


## 8. Decomposing a convertible — Example 25.8

Face 1000, 2% coupon, five years, conversion ratio 20; the stock at 40 with 35%
volatility, rates 4%, the issuer's credit spread putting the bond floor at a 7% yield.

**The convertible is a bond floor plus a call on 20 shares struck at 50.** Its delta walks
from 6 shares to 17 as the stock rises — bond-like when the stock is low, equity-like when
it is high — and the transition is what the issuer sold.


In [9]:
FACE, CPN, T_CB, RATIO = 1000.0, 0.02, 5.0, 20
S0, VOL, R_F, Y_CREDIT = 40.0, 0.35, 0.04, 0.07
STRIKE = FACE / RATIO


def bond_floor(y=Y_CREDIT):
    """Straight-bond value, continuously discounted at the issuer's yield."""
    return sum(FACE * CPN * exp(-y * t) for t in range(1, 6)) + FACE * exp(-y * T_CB)


def bs_call(S, K=STRIKE, r=R_F, sigma=VOL, T=T_CB):
    """Black-Scholes call, returning the price and N(d1)."""
    d1 = (log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    return S * N(d1) - K * exp(-r * T) * N(d2), N(d1)


floor = bond_floor()
call, nd1 = bs_call(S0)

show("bond floor", floor, 786.14)
show("conversion value (20 x 40)", RATIO * S0, 800.00)
show("conversion price (1000 / 20)", STRIKE, 50.00)
show("conversion premium over the share price, %", 100 * (STRIKE / S0 - 1), 25.0)
show("embedded call on 20 shares", RATIO * call, 237.13)
show("convertible value", floor + RATIO * call, 1023.28)
show("equity delta, shares", RATIO * nd1, 12.82)
print()

print("%12s %14s %12s %16s" % ("share price", "convertible", "parity", "delta (shares)"))
for S in (20, 30, 40, 50, 70):
    c, n1 = bs_call(float(S))
    print("%12d %14.2f %12.2f %16.2f"
          % (S, floor + RATIO * c, RATIO * S, RATIO * n1))


bond floor                                          786.1443   book    786.1400   ok
conversion value (20 x 40)                          800.0000   book    800.0000   ok
conversion price (1000 / 20)                         50.0000   book     50.0000   ok
conversion premium over the share price, %           25.0000   book     25.0000   ok
embedded call on 20 shares                          237.1347   book    237.1300   ok
convertible value                                  1023.2790   book   1023.2800   ok
equity delta, shares                                 12.8245   book     12.8200   ok

 share price    convertible       parity   delta (shares)
          20         827.87       400.00             6.00
          30         908.55       600.00             9.95
          40        1023.28       800.00            12.82
          50        1162.13      1000.00            14.82
          70        1484.96      1400.00            17.18


## 9. Pricing the convertible on a tree with default — Example 25.9

The additive split of Example 25.8 is a first look, not a valuation. Section 25.14 puts
the bond, its conversion right, the issuer's call and the issuer's default on one tree:
each step the stock moves up, down, or the issuer defaults with probability
$1 - e^{-\lambda\Delta t}$, in which case the stock is worthless and the bond is worth its
recovery. The rollback at every node is
$V = \max\{nS,\ \min(\text{continuation}, K_c)\}$ plus any coupon due.

**Three readings.** The default branch makes the embedded call *dearer*, not cheaper
(Merton's jump-to-ruin result: Black–Scholes–Merton at $r+\lambda$); the American
conversion right is worth nothing without dividends; and the issuer's call is the largest
term. The cell reproduces the five-step tree of Figure 25.4, the 500-step values, and the
three nodes the example works in full.


In [10]:
LAM_CB = 0.03 / (1 - 0.40)          # hazard rate from the credit triangle: s / (1 - R)
R_CB, K_CALL, CALL_FROM = 0.40, 1100.0, 2.0


def cb_tree(n, conv="american", call_price=None, call_from=CALL_FROM, trace=None):
    """Convertible on a tree where each step is up, down, or default (Hull's construction).

    conv: "none" (bond floor), "european" (convert only at maturity), "american".
    Rollback: V = max{ ratio*S, min(continuation, call_price) } + coupon due.
    trace: optional (step, S) -- returns that node's continuation value instead.
    """
    dt = T_CB / n
    u = exp(sqrt((VOL ** 2 - LAM_CB) * dt))
    d = 1.0 / u
    a = exp(R_F * dt)
    p_def = 1.0 - exp(-LAM_CB * dt)
    p_u = (a - d * exp(-LAM_CB * dt)) / (u - d)
    p_d = (u * exp(-LAM_CB * dt) - a) / (u - d)
    disc = exp(-R_F * dt)
    V = []
    for j in range(n + 1):
        S = S0 * u ** (n - 2 * j)
        V.append((max(RATIO * S, FACE) if conv != "none" else FACE) + CPN * FACE)
    for i in range(n - 1, -1, -1):
        t = i * dt
        coupon = CPN * FACE if (abs(t - round(t)) < 1e-9 and i > 0) else 0.0
        new_V = []
        for j in range(i + 1):
            S = S0 * u ** (i - 2 * j)
            cont = disc * (p_u * V[j] + p_d * V[j + 1] + p_def * R_CB * FACE)
            if trace is not None and i == trace[0] and abs(S - trace[1]) < 0.01:
                return cont
            conv_value = RATIO * S if conv != "none" else 0.0
            held = cont
            if call_price is not None and t >= call_from - 1e-9:
                held = max(min(cont, call_price), conv_value)
            v = max(held, conv_value) if conv == "american" else held
            new_V.append(v + coupon)
        V = new_V
    return V[0], dict(u=u, d=d, p_u=p_u, p_d=p_d, p_def=p_def, disc=disc)


_, prm = cb_tree(5)
for k, printed in (("u", 1.3090), ("d", 0.7639), ("p_u", 0.5763), ("p_d", 0.3749), ("p_def", 0.0488), ("disc", 0.9608)):
    show("five-step tree: " + k, prm[k], printed, tol=2e-3)
print()

print("%-52s %10s %10s" % ("", "5 steps", "500 steps"))
for label, kw, printed in (("bond floor (no conversion right)", dict(conv="none"), (793.50, 795.10)),
                           ("conversion at maturity only", dict(conv="european"), (1046.38, 1057.82)),
                           ("conversion at any time", dict(conv="american"), (1046.38, 1057.82)),
                           ("conversion at any time, callable at 1100 from year 2",
                            dict(conv="american", call_price=K_CALL), (994.47, 988.40))):
    v5, v500 = cb_tree(5, **kw)[0], cb_tree(500, **kw)[0]
    flag = "ok" if abs(v5 - printed[0]) < 0.01 and abs(v500 - printed[1]) < 0.01 else "DIFFERS"
    print("%-52s %10.2f %10.2f   %s" % (label, v5, v500, flag))
print()

show("node t=4, S=40: continuation", cb_tree(5, "american", K_CALL, trace=(4, 40.0)), 977.09)
show("node t=2, S=68.54: continuation (issuer calls, holder converts)", cb_tree(5, "american", K_CALL, trace=(2, 68.54)), 1426.81)
show("node t=3, S=52.36: continuation (issuer calls, holder takes 1100)", cb_tree(5, "american", K_CALL, trace=(3, 52.36)), 1148.00)


five-step tree: u                                     1.3090   book      1.3090   ok
five-step tree: d                                     0.7639   book      0.7639   ok
five-step tree: p_u                                   0.5763   book      0.5763   ok
five-step tree: p_d                                   0.3749   book      0.3749   ok
five-step tree: p_def                                 0.0488   book      0.0488   ok
five-step tree: disc                                  0.9608   book      0.9608   ok

                                                        5 steps  500 steps
bond floor (no conversion right)                         793.50     795.10   ok
conversion at maturity only                             1046.38    1057.82   ok
conversion at any time                                  1046.38    1057.82   ok
conversion at any time, callable at 1100 from year 2     994.47     988.40   ok

node t=4, S=40: continuation                        977.0859   book    977.0900   ok
node t=2

## 10. The correlation a hybrid needs and no market quotes — Example 25.10

A note paying a single coupon at maturity if the issuer survives **and** an equity index finishes above
a level $K$. The two legs are separately observable and their dependence is not, which is the whole
difficulty.

The index leg is struck at the note's initial level, so its marginal follows from the stated
parameters — $N(d_2) = 0.4261$ at a 4% rate, a 2% dividend yield and 30% volatility over five
years — and is computed here rather than chosen. The interesting quantity is the **joint**
probability. It is a bivariate normal, and the cell checks it against the three
things any such calculation must satisfy before its middle is believed: $\rho = 0$ must return the
product of the marginals, and $\rho \to \pm 1$ must attain the Fréchet bounds.

In [11]:
from scipy.stats import multivariate_normal

P_SURV, COUPON = 0.90, 8.0

# --- the index leg is struck at its initial level; its marginal follows from r, q and sigma
R_X, Q_X, SIG_X, T_X = 0.04, 0.02, 0.30, 5.0
D2_X = (R_X - Q_X - 0.5 * SIG_X ** 2) * T_X / (SIG_X * np.sqrt(T_X))
P_UP = N(D2_X)
show("d_2 for the index leg", D2_X, -0.1863)
show("P(index above its initial level)", P_UP, 0.4261)

# --- the joint probability, and the four controls that make it trustworthy
def both(rho, p_surv=P_SURV, p_up=P_UP):
    """P(survive AND index above) under a Gaussian copula on the two latent factors."""
    c, k = norm.ppf(1 - p_surv), norm.ppf(1 - p_up)
    cov = [[1.0, rho], [rho, 1.0]]
    # P(X > c, Y > k) by the survival function of the bivariate normal
    return multivariate_normal(mean=[0.0, 0.0], cov=cov).cdf([-c, -k])


INDEP = P_SURV * P_UP
show("control: rho = 0 gives the product of marginals", both(0.0), INDEP)
show("control: rho -> +1 gives min(p1, p2)", both(0.999), min(P_SURV, P_UP), tol=0.01)
show("control: rho -> -1 gives max(p1+p2-1, 0)", both(-0.999), max(P_SURV + P_UP - 1.0, 0.0), tol=0.01)
print()
print("%-8s %16s %18s %22s" % ("rho", "P(coupon paid)", "expected coupon", "effect on the coupon"))
for rho, p_book, cpn_book in ((-0.4, 0.3551, 2.8408), (0.0, 0.3835, 3.0678), (0.4, 0.4091, 3.2727)):
    p = both(rho)
    show("rho %+0.1f, P(coupon paid)" % rho, p, p_book)
    show("rho %+0.1f, expected coupon %%" % rho, COUPON * p, cpn_book)
    bp = 1e4 * COUPON * (p - INDEP) / 100.0
    print("%-8.1f %16.4f %18.4f %19.1f bp" % (rho, p, COUPON * p, bp))

# --- the band any dependence must respect, which is what makes a reserve defensible
lo, hi = max(P_SURV + P_UP - 1.0, 0.0), min(P_SURV, P_UP)
show("the Frechet band, bp of coupon", 1e4 * COUPON * (hi - lo) / 100.0, 80.0)
print()
print("Whatever the dependence, the coupon lies in [%.4f%%, %.4f%%] -- a band of %.0f bp."
      % (COUPON * lo, COUPON * hi, 1e4 * COUPON * (hi - lo) / 100.0))
print("A reserve sized to that width needs no agreement on the correlation itself.")

d_2 for the index leg                                -0.1863   book     -0.1863   ok
P(index above its initial level)                      0.4261   book      0.4261   ok


control: rho = 0 gives the product of marginals        0.3835   book      0.3835   ok
control: rho -> +1 gives min(p1, p2)                  0.4261   book      0.4261   ok
control: rho -> -1 gives max(p1+p2-1, 0)              0.3261   book      0.3261   ok

rho        P(coupon paid)    expected coupon   effect on the coupon
rho -0.4, P(coupon paid)                              0.3551   book      0.3551   ok
rho -0.4, expected coupon %                           2.8408   book      2.8408   ok
-0.4               0.3551             2.8408               -22.7 bp
rho +0.0, P(coupon paid)                              0.3835   book      0.3835   ok
rho +0.0, expected coupon %                           3.0678   book      3.0678   ok
0.0                0.3835             3.0678                -0.0 bp
rho +0.4, P(coupon paid)                              0.4091   book      0.4091   ok
rho +0.4, expected coupon %                           3.2727   book      3.2727   ok
0.4                0.4091    